#  Notebook 04 — Regression (Rainfall Amount)

Trains XGBoost + LightGBM regressors, evaluates RMSE/MAE/R², plots actual vs predicted.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
from config_utils import load_config
from train_regressor import run_training

warnings.filterwarnings('ignore')
cfg = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
xgb_reg, lgbm_reg, metrics = run_training(cfg['_meta']['config_path'])

## Results

In [ ]:
import json
with open(Path(cfg['paths']['reports']) / 'regressor_metrics.json') as f:
    m = json.load(f)
import pandas as pd
for model_name, model_metrics in m.items():
    print(f'\n=== {model_name.upper()} ===')
    print(pd.DataFrame(model_metrics))

## Actual vs Predicted Plot

In [ ]:
import pandas as pd, numpy as np, joblib
from features import build_features_for_splits, get_feature_columns, apply_scaler, prepare_Xy
from visualize import plot_actual_vs_predicted

train = pd.read_csv(cfg['paths']['train'], parse_dates=['date'])
val   = pd.read_csv(cfg['paths']['val'], parse_dates=['date'])
test  = pd.read_csv(cfg['paths']['test'], parse_dates=['date'])
_, _, test = build_features_for_splits(train=train, val=val, test=test)
feat_cols = get_feature_columns(cfg)
feat_cols = [c for c in feat_cols if c in test.columns]
reg = joblib.load(cfg['paths']['model_reg'])
sc  = joblib.load(cfg['paths']['scaler'].replace('.pkl','_reg.pkl'))
X_test_r, y_test_r = prepare_Xy(test, feat_cols, 'PRECTOTCORR', log_transform=True)
X_test_r_sc = apply_scaler(X_test_r, sc)
y_pred_log = reg.predict(X_test_r_sc)
y_true_mm = np.expm1(y_test_r)
y_pred_mm = np.expm1(y_pred_log).clip(min=0)
plot_actual_vs_predicted(y_true_mm, y_pred_mm, 'XGBoost Regressor', cfg)
print(f"Plot saved to {cfg['paths']['plots']}")